# MS2 — Data Exploration: EB-NeRD Click Prediction & Cold-Start Analysis

**Project Question:** *How well can we rank and predict clicks for new articles when historical interactions are sparse, and what's the best hybrid approach using content features + limited early signals?*

---

## Abstract

This project builds an end-to-end large-scale pipeline for native recommendation click prediction and ranking using the EB-NeRD dataset, with a focus on the cold-start problem for newly published content. Using Spark on SDSC Expanse, we transform impression logs into an impression–candidate training table, engineer time-aware features (to prevent label leakage), and generate cohorts that represent cold items (early life impressions) versus warm items (sufficient interaction history). We then train and tune CTR/ranking models using Ray, comparing interaction-only baselines, content-only models that generalize to unseen items, and hybrid approaches that blend content with early engagement signals. Evaluation emphasizes ranking quality (e.g., NDCG@k and MRR@k) and calibrated CTR prediction, reported separately for cold and warm items to quantify cold-start degradation and the lift provided by hybrid strategies.

---

## Dataset

**EB-NeRD (Ekstra Bladet News Recommendation Dataset)** — A large-scale public dataset released for the RecSys Challenge 2024. The **large** bundle spans six weeks of activity (Apr 27 – Jun 8, 2023) with:

- **~37.9 million** impression logs
- **~1.1 million** unique users
- **~125,000** news articles with rich metadata
- **~213 million** historical user interactions

Source: [recsys.eb.dk](https://recsys.eb.dk/) • License: General License Terms for academic research.

## Setup & Data Access

### Dataset Download

The EB-NeRD Large dataset is available at [https://recsys.eb.dk/dataset/](https://recsys.eb.dk/dataset/). You must accept the license agreement before downloading.

### On SDSC Expanse

```bash
# SSH into Expanse
ssh <username>@login.expanse.sdsc.edu

# Navigate to your project space (fast Lustre storage)
cd /expanse/lustre/projects/<allocation>/<username>

# Create data directory
mkdir -p ebnerd_data && cd ebnerd_data

# ── Core dataset ──
wget -O ebnerd_large.zip 'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_large.zip'
unzip ebnerd_large.zip -d ebnerd_large

# ── Embedding artifacts (separate downloads) ──
wget -O contrastive_vector.zip 'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/artifacts/Ekstra_Bladet_contrastive_vector.zip'
unzip contrastive_vector.zip -d artifacts

wget -O xlm_roberta.zip 'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/artifacts/FacebookAI_xlm_roberta_base.zip'
unzip xlm_roberta.zip -d artifacts

# ── (Optional) Test set ──
wget -O ebnerd_testset.zip 'https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_testset.zip'
unzip ebnerd_testset.zip -d ebnerd_testset
```

Expected directory structure after unzipping:

```
~/ebnerd_data/
├── ebnerd_large/
│   ├── train/
│   │   ├── behaviors.parquet
│   │   └── history.parquet
│   ├── validation/
│   │   ├── behaviors.parquet
│   │   └── history.parquet
│   └── articles.parquet
├── artifacts/
│   ├── Ekstra_Bladet_contrastive_vector/   ← contrastive embeddings (parquet inside)
│   └── FacebookAI_xlm_roberta_base/        ← XLM-RoBERTa embeddings (parquet inside)
└── ebnerd_testset/  (optional)
    ├── test/
    │   ├── behaviors.parquet
    │   └── history.parquet
    └── articles.parquet
```

> **Note:** The embedding artifacts are distributed as separate zip files, not inside `ebnerd_large.zip`. The exact parquet file/column names inside each artifact zip may vary — the loading cell below will auto-detect them.

---
## Configuration & Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import glob as globlib

DATA_ROOT = os.path.expanduser("~/ebnerd_data")
TRAIN_VAL_DIR = os.path.join(DATA_ROOT, "ebnerd_large")
TEST_DIR      = os.path.join(DATA_ROOT, "ebnerd_testset")
ARTIFACTS_DIR = os.path.join(DATA_ROOT, "artifacts")
CONTRASTIVE_DIR = os.path.join(ARTIFACTS_DIR, "Ekstra_Bladet_contrastive_vector")
ROBERTA_DIR     = os.path.join(ARTIFACTS_DIR, "FacebookAI_xlm_roberta_base")

spark = (
    SparkSession.builder
    .appName("EB-NeRD Data Exploration")
    .master("local[7]")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.instances", "7")
    .config("spark.sql.shuffle.partitions", 200)
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

---
## 1. Load the Dataset

In [ ]:
# Load core tables

df_behaviors_train = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "train", "behaviors.parquet"))
df_behaviors_val   = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "validation", "behaviors.parquet"))
df_history_train = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "train", "history.parquet"))
df_history_val   = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "validation", "history.parquet"))
df_articles = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "articles.parquet"))

# Load embedding artifacts (auto-detect parquet files inside artifact dirs)
def load_artifact(artifact_dir, label):
    candidates = globlib.glob(os.path.join(artifact_dir, "**", "*.parquet"), recursive=True)
    if not candidates:
        candidates = [artifact_dir]
    path = candidates[0]
    df = spark.read.parquet(path)
    print(f"{label}: loaded from {path}  (columns: {df.columns})")
    return df

df_contrastive = load_artifact(CONTRASTIVE_DIR, "Contrastive")
df_roberta     = load_artifact(ROBERTA_DIR,     "XLM-RoBERTa")

df_behaviors = df_behaviors_train.unionByName(df_behaviors_val, allowMissingColumns=True)

print("All datasets loaded.")

---
## 2. How Many Observations?

We have three core tables. Let's count rows in each.

In [ ]:
n_train     = df_behaviors_train.count()
n_val       = df_behaviors_val.count()
n_behaviors = n_train + n_val
n_articles  = df_articles.count()
n_history_train = df_history_train.count()
n_history_val   = df_history_val.count()
n_contrastive = df_contrastive.count()
n_roberta     = df_roberta.count()

print(f"Behaviors (train):       {n_train:>12,}")
print(f"Behaviors (validation):  {n_val:>12,}")
print(f"Behaviors (total):       {n_behaviors:>12,}")
print(f"Articles:                {n_articles:>12,}")
print(f"History (train users):   {n_history_train:>12,}")
print(f"History (val users):     {n_history_val:>12,}")
print(f"Contrastive embeddings:  {n_contrastive:>12,}")
print(f"RoBERTa embeddings:      {n_roberta:>12,}")
print()

n_users = df_behaviors.select("user_id").distinct().count()
print(f"Unique users in behaviors: {n_users:>10,}")
n_unique_articles = df_articles.select("article_id").distinct().count()
print(f"Unique articles:           {n_unique_articles:>10,}")

---
## 3. Describe All Columns

We examine each table's schema, data types, and distributions. For the EB-NeRD project, columns fall into three categories:

| Table | Purpose |
|---|---|
| **behaviors** | Impression logs — one row per page view. Contains the slate of candidates shown and which were clicked. |
| **articles** | Article metadata — title, body, category, sentiment, publication time, etc. |
| **history** | Per-user reading history prior to each data-split period. |

### 3.1 Behaviors Table

Each row is one **impression**, a user visited a page and was shown a slate of candidate articles.

| Column | Type | Scale | Description |
|---|---|---|---|
| `impression_id` | int | Nominal | Unique impression identifier |
| `article_id` | int (nullable) | Nominal | Article the user was reading when the impression occurred (null = frontpage) |
| `impression_time` | timestamp | Continuous | When the impression happened |
| `read_time` | float | Continuous | Seconds spent on the referring page |
| `scroll_percentage` | float (nullable) | Continuous | How far user scrolled (0–100) |
| `device_type` | int | Categorical | 1 = mobile, 2 = desktop, etc. |
| `article_ids_inview` | array[int] | Set/List | IDs of candidate articles shown in the recommendation widget |
| `article_ids_clicked` | array[int] | Set/List | Subset of inview articles that the user clicked |
| `user_id` | int | Nominal | User identifier |
| `is_sso_user` | boolean | Binary | Whether user logged in via SSO |
| `gender` | int (nullable) | Categorical | Gender code (nullable for anonymous users) |
| `postcode` | int (nullable) | Categorical | Postcode region code |
| `age` | int (nullable) | Ordinal | Age bucket |
| `is_subscriber` | boolean | Binary | Paid subscriber flag |
| `session_id` | int | Nominal | Session identifier |
| `next_read_time` | float | Continuous | Read time of the next page in session |
| `next_scroll_percentage` | float | Continuous | Scroll percentage of the next page |

In [ ]:
print("BEHAVIORS TABLE SCHEMA")
df_behaviors.printSchema()

In [ ]:
# Summary stats for continuous columns
df_behaviors.select("read_time", "scroll_percentage", "next_read_time", "next_scroll_percentage") \
    .describe().show()

In [ ]:
# Categorical column distributions
for col_name in ["device_type", "is_sso_user", "is_subscriber", "gender"]:
    print(f"{col_name} distribution:")
    df_behaviors.groupBy(col_name).agg(F.count("*").alias("count")) \
        .orderBy(F.desc("count")).show()

print("age distribution:")
df_behaviors.groupBy("age").agg(F.count("*").alias("count")) \
    .orderBy("age").show(20)

In [ ]:
# How many articles are shown/clicked per impression
print("Articles shown per impression:")
df_behaviors.select(F.size("article_ids_inview").alias("n_inview")).describe().show()

print("Articles clicked per impression:")
df_behaviors.select(F.size("article_ids_clicked").alias("n_clicked")).describe().show()

### 3.2 Articles Table

Each row is one **article** with rich metadata.

| Column | Type | Scale | Description |
|---|---|---|---|
| `article_id` | int | Nominal | Unique article identifier |
| `title` | string | Text | Headline |
| `subtitle` | string | Text | Sub-headline / abstract |
| `body` | string | Text | Full article body |
| `published_time` | timestamp | Continuous | Original publication time |
| `last_modified_time` | timestamp | Continuous | Last edit time |
| `premium` | boolean | Binary | Behind paywall? |
| `article_type` | string | Categorical | e.g., `article_default` |
| `category` | int | Categorical | Category code |
| `category_str` | string | Categorical | Human-readable category (e.g., `nyheder`, `sport`) |
| `subcategory` | array[int] | Categorical | Sub-category codes |
| `topics` | array[string] | Categorical/Text | Topic labels |
| `sentiment_score` | float | Continuous | Sentiment analysis score |
| `sentiment_label` | string | Categorical | `Positive`, `Negative`, `Neutral` |
| `total_inviews` | int (nullable) | Continuous | Total times shown (lifetime) |
| `total_pageviews` | int (nullable) | Continuous | Total page views (lifetime) |
| `total_read_time` | float (nullable) | Continuous | Total read time (lifetime) |
| `ner_clusters` | array[string] | Text | Named entities |
| `entity_groups` | array[string] | Categorical | Entity types (PER, ORG, etc.) |
| `image_ids` | array[long] | Nominal | Associated image IDs |
| `url` | string | Nominal | Article URL |

### 3.2b Embedding Artifacts (Separate Downloads)

Pre-computed dense vector representations for each article, distributed as **separate zip files** from the [EB-NeRD artifacts](https://recsys.eb.dk/dataset/):

| Artifact | Download | Description |
|---|---|---|
| **Ekstra_Bladet_contrastive_vector** | [341 MB zip](https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/artifacts/Ekstra_Bladet_contrastive_vector.zip) | Embeddings learned via contrastive learning on user–article interactions |
| **FacebookAI_xlm_roberta_base** | [341 MB zip](https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/artifacts/FacebookAI_xlm_roberta_base.zip) | Embeddings from XLM-RoBERTa language model fine-tuned on article text |

Each contains a parquet file keyed by `article_id` with a dense float vector column.

These are **critical for cold-start prediction** — when an article has zero interaction history, its embedding vector is the primary representation available.

> **Also available** (not used in this project): word2vec embeddings (133 MB), image embeddings (372 MB), google-bert-base-multilingual (344 MB).

In [ ]:
print("ARTICLES TABLE SCHEMA")
df_articles.printSchema()

In [ ]:
# Summary stats for continuous columns
df_articles.select("sentiment_score", "total_inviews", "total_pageviews", "total_read_time") \
    .describe().show()

In [ ]:
# Categorical distributions
for col_name in ["category_str", "sentiment_label", "article_type", "premium"]:
    print(f"{col_name} distribution:")
    df_articles.groupBy(col_name).agg(F.count("*").alias("count")) \
        .orderBy(F.desc("count")).show(20)

In [ ]:
# ── Text length statistics (title, subtitle, body word counts) ──
df_text_stats = df_articles.select(
    F.size(F.split(F.col("title"), "\\s+")).alias("title_words"),
    F.size(F.split(F.col("subtitle"), "\\s+")).alias("subtitle_words"),
    F.size(F.split(F.col("body"), "\\s+")).alias("body_words")
)
print("Text lengths (word counts):")
df_text_stats.describe().show()

In [ ]:
def explore_embedding(df, name):
    print("=" * 60)
    print(f"{name} embeddings")
    df.printSchema()
    vec_col = [c for c in df.columns if c != "article_id"][0]
    dim = df.select(F.size(F.col(vec_col)).alias("dim")).first()["dim"]
    print(f"Dimension: {dim}")
    print(f"Rows: {df.count():,}")
    n_art = df_articles.select("article_id").distinct().count()
    n_match = df_articles.join(df, "article_id", "inner").count()
    print(f"Coverage: {n_match:,} / {n_art:,} ({100*n_match/n_art:.1f}%)")
    n_null = df.filter(F.col(vec_col).isNull()).count()
    print(f"Null vectors: {n_null}")
    print("First element stats:")
    df.select(F.col(vec_col)[0].alias("elem_0")).describe().show()
    print()

explore_embedding(df_contrastive, "Contrastive")
explore_embedding(df_roberta, "XLM-RoBERTa")

### 3.3 History Table

Each row is one **user** with their reading history from the prior period.

| Column | Type | Scale | Description |
|---|---|---|---|
| `user_id` | int | Nominal | User identifier |
| `article_id_fixed` | array[int] | List | Article IDs the user interacted with (chronological) |
| `impression_time_fixed` | array[timestamp] | List | Timestamps of interactions |
| `scroll_percentage_fixed` | array[float] | List | Scroll depth per interaction |
| `read_time_fixed` | array[float] | List | Read time per interaction |

In [ ]:
# ── Schema ──
print("=" * 60)
print("HISTORY TABLE SCHEMA")
print("=" * 60)
df_history_train.printSchema()

In [ ]:
# ── History length statistics ──
print("Number of historical interactions per user:")
df_history_train.select(
    F.size("article_id_fixed").alias("history_length")
).describe().show()

### 3.4 Target Variable

The **target** for our CTR/ranking task is binary:

- For each impression, `article_ids_inview` lists the candidate articles shown.
- `article_ids_clicked` lists which of those the user clicked.
- When we build the training table (in MS3), each **(impression, candidate)** pair gets a label:
  - **1** if the candidate is in `article_ids_clicked`
  - **0** otherwise

Let's compute the overall click-through statistics.

In [ ]:
# ── Target variable: positives vs negatives ──
target_stats = df_behaviors.select(
    F.size("article_ids_clicked").alias("n_pos"),
    F.size("article_ids_inview").alias("n_inview")
).select(
    F.sum("n_pos").alias("total_positives"),
    F.sum("n_inview").alias("total_candidates"),
    (F.sum("n_inview") - F.sum("n_pos")).alias("total_negatives")
).collect()[0]

total_pos = target_stats["total_positives"]
total_neg = target_stats["total_negatives"]
total_cand = target_stats["total_candidates"]
np_ratio = total_neg / total_pos if total_pos > 0 else float('inf')
ctr = total_pos / total_cand if total_cand > 0 else 0

print(f"Total candidates (impression × inview):  {total_cand:>14,}")
print(f"Total positives (clicks):                {total_pos:>14,}")
print(f"Total negatives (non-clicks):            {total_neg:>14,}")
print(f"Negative-to-positive ratio:              {np_ratio:>14.2f}")
print(f"Overall CTR:                             {ctr:>14.4f}  ({ctr*100:.2f}%)")
print()
print("==> This is a heavily IMBALANCED classification problem.")
print("    The NP-ratio tells us roughly how many negatives per positive in the exploded training table.")

---
## 4. Missing and Duplicate Values

In [ ]:
# ============================================================
# MISSING VALUES — Behaviors
# ============================================================
print("=" * 60)
print("MISSING VALUES — Behaviors Table")
print("=" * 60)

behaviors_null_counts = df_behaviors.select(
    *[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_behaviors.columns
    ]
)
behaviors_null_counts.show(truncate=False)

In [ ]:
# ── Missing values as percentages ──
total_rows = n_behaviors
null_row = behaviors_null_counts.collect()[0]

print(f"{'Column':<30} {'Nulls':>12} {'Pct':>8}")
print("-" * 52)
for c in df_behaviors.columns:
    nulls = null_row[c]
    pct = 100 * nulls / total_rows if total_rows > 0 else 0
    flag = " *** " if pct > 0 else ""
    print(f"{c:<30} {nulls:>12,} {pct:>7.2f}%{flag}")

In [ ]:
# ============================================================
# MISSING VALUES — Articles
# ============================================================
print("=" * 60)
print("MISSING VALUES — Articles Table")
print("=" * 60)

articles_null_counts = df_articles.select(
    *[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_articles.columns
    ]
)

null_row_art = articles_null_counts.collect()[0]
print(f"{'Column':<30} {'Nulls':>12} {'Pct':>8}")
print("-" * 52)
for c in df_articles.columns:
    nulls = null_row_art[c]
    pct = 100 * nulls / n_articles if n_articles > 0 else 0
    flag = " *** " if pct > 0 else ""
    print(f"{c:<30} {nulls:>12,} {pct:>7.2f}%{flag}")

In [ ]:
# ============================================================
# DUPLICATE VALUES
# ============================================================
print("=" * 60)
print("DUPLICATE CHECK")
print("=" * 60)

# Behaviors: duplicates by impression_id
n_distinct_impressions = df_behaviors.select("impression_id").distinct().count()
print(f"Behaviors total rows:        {n_behaviors:>12,}")
print(f"Distinct impression_ids:     {n_distinct_impressions:>12,}")
print(f"Duplicate impressions:       {n_behaviors - n_distinct_impressions:>12,}")
print()

# Articles: duplicates by article_id
n_distinct_articles = df_articles.select("article_id").distinct().count()
print(f"Articles total rows:         {n_articles:>12,}")
print(f"Distinct article_ids:        {n_distinct_articles:>12,}")
print(f"Duplicate articles:          {n_articles - n_distinct_articles:>12,}")

---
## 5. Data Visualizations

All aggregations are performed using **Spark DataFrames**. We collect only the small, aggregated results to the driver for plotting with matplotlib.

In [ ]:
# ── Plotting defaults ──
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 100,
})

### Plot 1 — Article Category Distribution (Bar Chart)

Shows how articles are distributed across news categories.

In [ ]:
# Spark aggregation
cat_dist = (
    df_articles
    .groupBy("category_str")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .collect()
)

categories = [row["category_str"] for row in cat_dist]
counts     = [row["count"] for row in cat_dist]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(categories, counts, color="steelblue", edgecolor="white")
ax.set_title("Article Category Distribution")
ax.set_xlabel("Category")
ax.set_ylabel("Number of Articles")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
plt.tight_layout()
plt.show()

print("\nInsight: The dataset is dominated by 'nyheder' (news), 'underholdning' (entertainment),")
print("'krimi' (crime), and 'sport'. This mirrors real tabloid content distribution.")
print("Smaller categories like 'vin' (wine) and 'opinionen' (opinion) have much fewer articles.")

### Plot 2 — Number of Candidates per Impression (Histogram)

How many articles does the recommendation widget show per impression?

In [ ]:
# Spark aggregation — count of impressions by inview list size
inview_dist = (
    df_behaviors
    .select(F.size("article_ids_inview").alias("n_inview"))
    .groupBy("n_inview")
    .agg(F.count("*").alias("count"))
    .filter(F.col("n_inview") <= 40)  # cap for readability
    .orderBy("n_inview")
    .collect()
)

x_vals = [row["n_inview"] for row in inview_dist]
y_vals = [row["count"] for row in inview_dist]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x_vals, y_vals, color="darkorange", edgecolor="white", width=0.8)
ax.set_title("Number of Candidate Articles per Impression")
ax.set_xlabel("Candidates Shown (article_ids_inview length)")
ax.set_ylabel("Number of Impressions")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

print("\nInsight: Most impressions show between 5 and 15 candidates.")
print("The median is around 9 candidates per impression.")
print("This is the key dimension — each impression will explode into n_inview rows in the training table.")

### Plot 3 — Read Time Distribution (Histogram)

How long do users spend on the referring page before viewing the recommendation widget?

In [ ]:
# Spark aggregation — bucket read_time into bins
BIN_WIDTH = 5  # seconds
MAX_READ = 240  # cap at 4 minutes

read_time_dist = (
    df_behaviors
    .filter(F.col("read_time").isNotNull())
    .filter(F.col("read_time") < MAX_READ)
    .select((F.floor(F.col("read_time") / BIN_WIDTH) * BIN_WIDTH).cast("int").alias("bin"))
    .groupBy("bin")
    .agg(F.count("*").alias("count"))
    .orderBy("bin")
    .collect()
)

x_vals = [row["bin"] for row in read_time_dist]
y_vals = [row["count"] for row in read_time_dist]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x_vals, y_vals, width=BIN_WIDTH * 0.9, color="seagreen", edgecolor="white")
ax.set_title("Distribution of Read Time (seconds)")
ax.set_xlabel("Read Time (s)")
ax.set_ylabel("Number of Impressions")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

print("\nInsight: Read time is heavily right-skewed. Most impressions have short read times (<60s).")
print("This suggests many users quickly scan the frontpage/article and see the widget.")
print("Read time is a useful behavioral feature — longer reads may indicate higher engagement.")

### Plot 4 — Click-Through Rate by Category (Bar Chart)

Which article categories get clicked more often when shown?

In [ ]:
# Step 1: Explode inview articles and mark which were clicked
# We do this on a SAMPLE to keep the computation fast during exploration.
df_sampled = df_behaviors.sample(fraction=0.05, seed=42)

df_exploded = (
    df_sampled
    .select(
        F.col("impression_id"),
        F.explode("article_ids_inview").alias("candidate_id"),
        F.col("article_ids_clicked")
    )
    .withColumn(
        "clicked",
        F.when(F.array_contains(F.col("article_ids_clicked"), F.col("candidate_id")), 1).otherwise(0)
    )
)

# Step 2: Join with articles to get category
df_ctr_by_cat = (
    df_exploded
    .join(df_articles.select("article_id", "category_str"), 
          df_exploded["candidate_id"] == df_articles["article_id"], "left")
    .groupBy("category_str")
    .agg(
        F.sum("clicked").alias("clicks"),
        F.count("*").alias("impressions")
    )
    .withColumn("ctr", F.col("clicks") / F.col("impressions"))
    .filter(F.col("category_str").isNotNull())
    .orderBy(F.desc("ctr"))
    .collect()
)

cats = [row["category_str"] for row in df_ctr_by_cat]
ctrs = [row["ctr"] for row in df_ctr_by_cat]
imps = [row["impressions"] for row in df_ctr_by_cat]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(cats, ctrs, color="crimson", edgecolor="white")
ax.set_title("Click-Through Rate by Article Category (sampled 5%)")
ax.set_xlabel("Category")
ax.set_ylabel("CTR")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.show()

print("\nInsight: CTR varies significantly across categories.")
print("This is a strong signal — category is a content-only feature available even for cold-start items.")

### Plot 5 — Device Type Distribution (Bar Chart)

How do impression counts differ by device?

In [ ]:
# Spark aggregation
device_dist = (
    df_behaviors
    .groupBy("device_type")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .collect()
)

device_labels = [f"Type {row['device_type']}" for row in device_dist]
device_counts = [row["count"] for row in device_dist]

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(device_labels, device_counts, color=["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"][:len(device_labels)],
       edgecolor="white")
ax.set_title("Impressions by Device Type")
ax.set_xlabel("Device Type")
ax.set_ylabel("Number of Impressions")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

print("\nInsight: Device type is a contextual feature. If most traffic is mobile,")
print("the widget layout / click behavior may differ from desktop. Useful as a model feature.")

### Plot 6 — Sentiment Score Distribution (Histogram)

Distribution of sentiment scores across the article catalog.

In [ ]:
# Spark aggregation — bucket sentiment_score
SENT_BIN = 0.05
sentiment_dist = (
    df_articles
    .filter(F.col("sentiment_score").isNotNull())
    .select((F.floor(F.col("sentiment_score") / SENT_BIN) * SENT_BIN).alias("bin"))
    .groupBy("bin")
    .agg(F.count("*").alias("count"))
    .orderBy("bin")
    .collect()
)

x_vals = [float(row["bin"]) for row in sentiment_dist]
y_vals = [row["count"] for row in sentiment_dist]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x_vals, y_vals, width=SENT_BIN * 0.9, color="mediumpurple", edgecolor="white")
ax.set_title("Article Sentiment Score Distribution")
ax.set_xlabel("Sentiment Score")
ax.set_ylabel("Number of Articles")
plt.tight_layout()
plt.show()

print("\nInsight: Sentiment scores cluster at the extremes — most articles score very close to 0 or 1.")
print("This is a content-based feature available at publish time (cold-start friendly).")

### Plot 7 — Cold-Start Profiling: Article Impression Count Distribution (Log-Scale Histogram)

This is critical for our project — it shows how many articles have very few impressions (cold items) vs. many (warm items).

In [ ]:
# Count how many times each article appeared as a candidate across all impressions
article_impression_counts = (
    df_behaviors
    .select(F.explode("article_ids_inview").alias("article_id"))
    .groupBy("article_id")
    .agg(F.count("*").alias("n_impressions"))
)

# Bucket into log-scale bins for visualization
impression_buckets = (
    article_impression_counts
    .withColumn("log_bucket", F.floor(F.log10(F.col("n_impressions"))).cast("int"))
    .groupBy("log_bucket")
    .agg(
        F.count("*").alias("n_articles"),
        F.min("n_impressions").alias("min_imps"),
        F.max("n_impressions").alias("max_imps")
    )
    .orderBy("log_bucket")
    .collect()
)

bucket_labels = [f"{row['min_imps']}-{row['max_imps']}" for row in impression_buckets]
n_arts = [row["n_articles"] for row in impression_buckets]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(bucket_labels, n_arts, color="teal", edgecolor="white")
ax.set_title("Articles by Total Impression Count (log-scale buckets)")
ax.set_xlabel("Impression Count Range")
ax.set_ylabel("Number of Articles")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print("\nInsight: Many articles have very few impressions — these are the COLD-START items.")
print("Articles in the lowest bucket(s) have minimal interaction history,")
print("so interaction-only models will fail on them. Content features are critical here.")

In [ ]:
# ── Cold vs Warm split statistics ──
# Define 'cold' as articles with <= 50 impressions, 'warm' as > 50
COLD_THRESHOLD = 50

cold_warm = (
    article_impression_counts
    .withColumn(
        "cohort",
        F.when(F.col("n_impressions") <= COLD_THRESHOLD, "cold").otherwise("warm")
    )
    .groupBy("cohort")
    .agg(
        F.count("*").alias("n_articles"),
        F.mean("n_impressions").alias("avg_impressions"),
        F.sum("n_impressions").alias("total_impressions")
    )
)

print(f"Cold-start threshold: <= {COLD_THRESHOLD} impressions")
cold_warm.show()

print("This tells us what fraction of the article catalog is 'cold' and how much")
print("of total impression traffic they account for — the key tension in our project.")

### Plot 8 — Scatter Plot: Article Age vs. Total Pageviews

Shows the relationship between how old an article is and its engagement. Newer articles (right side) with few pageviews are the cold-start challenge.

In [ ]:
# Compute article age in days (relative to the end of the dataset period)
DATASET_END = "2023-06-08"

age_views = (
    df_articles
    .filter(F.col("published_time").isNotNull())
    .filter(F.col("total_pageviews").isNotNull())
    .select(
        F.datediff(F.lit(DATASET_END), F.col("published_time")).alias("age_days"),
        F.col("total_pageviews")
    )
    .filter(F.col("age_days").between(0, 60))  # focus on the 6-week window
    .sample(fraction=0.3, seed=42)  # sample for plotting
    .collect()
)

ages = [row["age_days"] for row in age_views]
views = [row["total_pageviews"] for row in age_views]

fig, ax = plt.subplots(figsize=(14, 6))
ax.scatter(ages, views, alpha=0.3, s=8, color="coral")
ax.set_title("Article Age vs. Total Pageviews (6-week window)")
ax.set_xlabel("Article Age (days since published)")
ax.set_ylabel("Total Pageviews")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

print("\nInsight: Very new articles (age < 3 days) span a huge range of pageviews.")
print("Some go viral quickly, but most have low engagement — the cold-start problem.")
print("Older articles tend to have more accumulated pageviews, making interaction features reliable.")

---
## 6. Preprocessing Plan

The following preprocessing steps will be implemented in **MS3** using PySpark on SDSC Expanse.

### 6.1 Handling Missing Values

| Column | Strategy |
|---|---|
| `article_id` (behaviors) | **Keep nulls** — null means the user was on the frontpage; we'll encode this as a boolean feature `is_frontpage`. |
| `scroll_percentage` | **Impute with 0** or **median** — null likely means the user didn't scroll (engagement signal). |
| `gender`, `age`, `postcode` | **Encode as a separate category** (e.g., -1 or "unknown") — these are null for anonymous/non-SSO users, which is informative in itself. |
| `total_inviews`, `total_pageviews`, `total_read_time` (articles) | **Impute with 0** — null means the article has no recorded lifetime stats (likely very new → cold-start signal). |
| `next_read_time`, `next_scroll_percentage` | **Impute with 0** — null means the user left after this impression. |

**Spark operations:** `F.when().otherwise()`, `F.coalesce()`, `df.na.fill()`

### 6.2 Handling Data Imbalance

The target is heavily imbalanced (~10:1 negative-to-positive ratio). We will:

1. **Negative downsampling (Wu et al. 2019 strategy):** For each impression, keep all positive candidates and sample `npratio` negatives (e.g., npratio=4). This is the standard approach in news recommendation.
2. **No SMOTE/oversampling** — the dataset is large enough, and ranking metrics (NDCG, MRR) naturally handle imbalance better than classification metrics.
3. **Stratified train/test split** by time (temporal split, not random) to prevent data leakage.

**Spark operations:** `F.explode()`, `F.array_contains()`, window functions with `F.row_number()`, `df.sample()`

### 6.3 Feature Engineering & Transformations

#### Content-Based Features (available at publish time — cold-start friendly)
- **Pre-trained embeddings:** `contrastive_vector` and `roberta_vector` joined to each candidate by `article_id` — these are the **primary cold-start features** since they exist for every article at publish time with no interaction history needed
- **Category one-hot encoding:** `category_str` → `StringIndexer` + `OneHotEncoder`
- **Text length features:** word counts for title, subtitle, body
- **Sentiment features:** `sentiment_score`, `sentiment_label` encoded
- **Time-of-day / day-of-week:** from `published_time` and `impression_time`
- **Premium flag:** boolean feature
- **Topic features:** explode `topics` array → multi-hot encoding or hashing

#### Interaction-Based Features (require history — NOT available for cold items)
- **Rolling CTR:** article's click-through rate over last 1h / 6h / 24h windows *before* impression time (to prevent leakage)
- **Rolling popularity:** impression count over recent windows
- **Recency-weighted engagement:** time-decayed read_time and scroll_percentage averages

**Spark operations:** Window functions (`Window.partitionBy().orderBy().rangeBetween()`), `F.unix_timestamp()`, `pyspark.ml.feature.StringIndexer`, `OneHotEncoder`, `VectorAssembler`

#### User-Level Features
- **History length:** number of articles in user's history
- **Category preference profile:** distribution of categories in user's history
- **Average read time / scroll depth** from history
- **SSO / subscriber / demographics** features

**Spark operations:** `F.size()`, `F.array_distinct()`, `df.join()`, `groupBy().agg()`

### 6.4 Cold-Start Cohort Creation

We define cold-start articles using multiple criteria:
1. **Time-based:** Articles published within the first N hours of the dataset period
2. **Impression count-based:** Articles with ≤ X total impressions at prediction time
3. **Hybrid:** First-seen in logs (for articles without reliable `published_time`)

Models will be evaluated **separately** on cold vs. warm cohorts to quantify degradation.

**Spark operations:** `F.datediff()`, `F.unix_timestamp()`, window functions for cumulative counts, `df.withColumn().when()`

### 6.5 Training Table Construction (Impression × Candidate Explosion)

The key Spark-intensive step:

```
raw impressions (~38M rows)  →  explode article_ids_inview  →  ~440M candidate rows
```

Each row: `(impression_id, user_id, candidate_article_id, clicked, features...)`

This is where Spark shines — the explosion + time-windowed aggregate joins across weeks of data require distributed memory, parallel I/O, and shuffle-intensive operations that exceed typical laptop resources.

**Spark operations:** `F.explode()`, `F.array_contains()`, `df.join()`, `df.repartition()`, `df.write.parquet()` with partitioning

---
## Cleanup

In [ ]:
# Stop the Spark session
spark.stop()
print("Spark session stopped.")